In [13]:
import numpy as np
import open3d as o3d
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("./coche_coche_moto.csv")

original_points = df[["x", "y", "z"]].to_numpy(dtype=np.float64)
original_points = original_points[np.linalg.norm(original_points, axis=1) > 0.1]
points = original_points.copy()

# Obtener los ejes usados para delimitar la region de la carretera.
x = points[:, 0]
y = points[:, 1]
z = points[:, 2]

# Limites robustos para eliminar los puntos que quedan fuera de la nube.
xmin, xmax = np.percentile(x, [0, 100])
ymin, ymax = np.percentile(y, [3, 100])
zmin, zmax = np.percentile(z, [20, 85])

mask = (
    (x >= xmin) & (x <= xmax) &
    (y >= ymin) & (y <= ymax) &
    (z >= zmin) & (z <= zmax)
)
points = points[mask]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Elimina mediciones aisladas antes de buscar el plano de la carretera.
pcd, inlier_indices = pcd.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=2.0
)
pcd_down = pcd.voxel_down_sample(0.08)

plane_model, ground_indices = pcd_down.segment_plane(
    distance_threshold=0.15,
    ransac_n=3,
    num_iterations=500
)

ground = pcd_down.select_by_index(ground_indices)
objects = pcd_down.select_by_index(ground_indices, invert=True)

print("Puntos originales:", len(original_points))
print("Puntos tras el filtro espacial:", len(points))
print("Puntos reducidos:", len(pcd_down.points))
print("Puntos carretera:", len(ground.points))
print("Puntos objetos:", len(objects.points))

labels = np.array(
    objects.cluster_dbscan(
        eps=0.6,
        min_points=10,
        print_progress=True
    )
)

max_label = labels.max() if len(labels) else -1
print("Clusters encontrados:", max_label + 1)

for i in range(max_label + 1):
    cantidad = np.sum(labels == i)
    print("Cluster", i, ":", cantidad, "puntos")

# Colores base para los clusters y gris para puntos sin cluster.
cluster_colors = plt.get_cmap("tab20")(
    np.linspace(0, 1, max(max_label + 1, 1))
)[:, :3]

# Pinta todos los puntos originales. Los cercanos a un objeto reciben su color.
original_colors = np.full((len(original_points), 3), [0.55, 0.55, 0.55])
if len(objects.points) > 0 and max_label >= 0:
    object_points = np.asarray(objects.points)
    object_tree = o3d.geometry.KDTreeFlann(objects)
    max_color_distance = 0.25

    for point_index, point in enumerate(original_points):
        count, neighbor_indices, neighbor_distances = object_tree.search_knn_vector_3d(
            point, 1
        )
        if count > 0 and neighbor_distances[0] <= max_color_distance ** 2:
            nearest_object_index = neighbor_indices[0]
            object_label = labels[nearest_object_index]
            if object_label >= 0:
                original_colors[point_index] = cluster_colors[object_label]

original_cloud = o3d.geometry.PointCloud()
original_cloud.points = o3d.utility.Vector3dVector(original_points)
original_cloud.colors = o3d.utility.Vector3dVector(original_colors)

# Cajas alineadas con los ejes para cada cluster, sin incluir el ruido.
bounding_boxes = []
for cluster_id in range(max_label + 1):
    cluster_indices = np.where(labels == cluster_id)[0]
    cluster_cloud = objects.select_by_index(cluster_indices)
    bounding_box = cluster_cloud.get_axis_aligned_bounding_box()
    bounding_box.color = cluster_colors[cluster_id]
    bounding_boxes.append(bounding_box)

print("Bounding boxes creadas:", len(bounding_boxes))

# Ejes: X rojo, Y verde y Z azul.
axis_size = max(1.0, 0.1 * np.ptp(original_points, axis=0).max())
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=axis_size,
    origin=[0, 0, 0]
)

o3d.visualization.draw_geometries(
    [original_cloud, *bounding_boxes],
    zoom=0.5,
    front=[-0.4999, -0.1659, -0.8499],
    lookat=[2.1813, 2.0619, 2.0999],
    up=[0.1204, -0.9852, 0.1215]
)

Puntos originales: 56961
Puntos tras el filtro espacial: 35417
Puntos reducidos: 24792
Puntos carretera: 22438
Puntos objetos: 2354
Clusters encontrados: 3
Cluster 0 : 926 puntos
Cluster 1 : 1406 puntos
Cluster 2 : 11 puntos
Bounding boxes creadas: 3
[Open3D WARNING] [ViewControl] SetViewPoint() failed because window height and width are not set.
